In [2]:
# ============================================================
# NOTEBOOK 06 – Agentic Reasoning Loop
# Multi-turn financial advisor agent with session memory
# Requires GROQ_API_KEY set in .env
# ============================================================
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

print('✅ Project root:', PROJECT_ROOT)


✅ Project root: C:\Users\SomnathYadav\PycharmProjects\virtual-financial-advisor


In [11]:
# ----------------------------------------------------------
# Cell 1 – Create agent session
# create_session() is preferred; create_agent() is an alias
# ----------------------------------------------------------
from src.agent.agent_core import create_session

USER_ID = 'user_1'
session = create_session(USER_ID)
print(f'Agent session created for user: {USER_ID}')


Agent session created for user: user_1


In [4]:
# ----------------------------------------------------------
# Cell 2 – Single interactive question (invoke shim)
# Compatible with old agent.run() / agent.invoke() calls
# ----------------------------------------------------------
response = session.invoke({'input': 'Analyze my finances and tell me how I can save more'})
print('=== Agent Response ===')
print(response['output'])


=== Agent Response ===
Based on the analysis, you have a total income of $165,306.82 and total expenses of $58,181.77, with a savings rate of 0.65. Your category-wise expense breakdown shows that you spend the most on 'Education' ($7,619.06) and 'Transport' ($9,443.71).

The risky pattern analysis did not reveal any risky financial patterns.

The expense cut simulation suggests that if you cut your monthly expenses by 15% for 12 months, you can save an extra $7,272.73 per year, with a new savings rate of 0.7.

The saving more simulation suggests that if you save 10% more of your monthly income for 12 months, you can save an extra $16,530.68.

To save more, I recommend that you:

1. Cut back on discretionary spending, such as 'Entertainment' ($5,041.22) and 'Dining' ($5,192.37), by 15% to save an extra $727.27 per month.
2. Allocate the extra savings towards your savings goals, such as building an emergency fund or paying off high-interest debt.
3. Consider increasing your income by 10%

In [5]:
# ----------------------------------------------------------
# Cell 3 – Multi-turn chat (maintains session context)
# ----------------------------------------------------------
questions = [
    'What is my current savings rate and biggest expense category?',
    'If I cut my dining expenses by 20%, how much would I save in a year?',
    'Are there any risky financial patterns I should be aware of?',
]

for q in questions:
    print(f'\nQ: {q}')
    print(f'A: {session.chat(q)}')



Q: What is my current savings rate and biggest expense category?
A: Your current savings rate is 0.65 (or 65%) and your biggest expense category is 'Transport' with a total spend of $9,443.71.

Q: If I cut my dining expenses by 20%, how much would I save in a year?
A: If you cut your dining expenses by 20%, you would save $1,038.47 per year. This is calculated by taking 20% of your current annual dining expenses, which is $5,192.37.

Q: Are there any risky financial patterns I should be aware of?
A: There are no risky financial patterns detected in your financial data. This suggests that you have a relatively healthy financial situation, with no signs of overspending, low savings rate, idle savings, or unusually large individual expenses. However, it's always a good idea to continue monitoring your finances and making adjustments as needed to maintain a stable financial situation.


In [6]:
# ----------------------------------------------------------
# Cell 4 – Full autonomous 5-step analysis
# Agent calls ALL tools by itself without being prompted
# ----------------------------------------------------------
print('=== Full Autonomous Financial Analysis ===\n')
report = session.analyze()
print(report)


=== Full Autonomous Financial Analysis ===

Based on the analysis, here are three prioritized recommendations with specific dollar amounts, ordered by highest financial impact:

1. Invest your annual net savings of $107,125.05 at a 7% annual return for 10 years, which can grow to a projected portfolio value of $1,480,087.68, earning a total interest of $408,837.18.
2. Cut your monthly expenses by 15% for 12 months, which can result in a total savings of $8,727.27 and increase your savings rate to 0.7. This can be achieved by reducing your expenses in categories such as 'Entertainment' ($5,041.22) and 'Dining' ($5,192.37).
3. Increase your savings rate by 10% for 12 months, which can result in an extra savings of $16,530.68. This can be achieved by allocating a larger portion of your income towards savings and investments.

These recommendations are prioritized based on their potential financial impact, with the investment opportunity having the highest potential return, followed by exp

In [7]:
# ----------------------------------------------------------
# Cell 5 – Follow-up after analysis (context is preserved)
# ----------------------------------------------------------
follow_up = 'Based on the analysis, which single action would have the biggest impact?'
print(f'Q: {follow_up}')
print(f'A: {session.chat(follow_up)}')


Q: Based on the analysis, which single action would have the biggest impact?
A: Based on the analysis, the single action that would have the biggest impact is investing your annual net savings of $107,125.05 at a 7% annual return for 10 years. This can grow to a projected portfolio value of $1,480,087.68, earning a total interest of $408,837.18.

This action has the biggest impact because it not only saves you money but also grows your wealth over time through the power of compound interest. The potential return on investment is significantly higher than the other options, making it the most impactful single action you can take.


In [8]:
# ----------------------------------------------------------
# Cell 6 – Session metadata & conversation history
# ----------------------------------------------------------
print(f'\n--- Session Summary ---')
print(f'User ID    : {session.user_id}')
print(f'Turn Count : {session.turn_count}')
print()
for i, turn in enumerate(session.history, 1):
    role    = turn['role'].upper()
    snippet = turn['content'][:150].replace('\n', ' ')
    suffix  = '...' if len(turn['content']) > 150 else ''
    print(f'[{i}] {role}: {snippet}{suffix}')



--- Session Summary ---
User ID    : user_1
Turn Count : 6

[1] USER: Analyze my finances and tell me how I can save more
[2] ASSISTANT: Based on the analysis, you have a total income of $165,306.82 and total expenses of $58,181.77, with a savings rate of 0.65. Your category-wise expens...
[3] USER: What is my current savings rate and biggest expense category?
[4] ASSISTANT: Your current savings rate is 0.65 (or 65%) and your biggest expense category is 'Transport' with a total spend of $9,443.71.
[5] USER: If I cut my dining expenses by 20%, how much would I save in a year?
[6] ASSISTANT: If you cut your dining expenses by 20%, you would save $1,038.47 per year. This is calculated by taking 20% of your current annual dining expenses, wh...
[7] USER: Are there any risky financial patterns I should be aware of?
[8] ASSISTANT: There are no risky financial patterns detected in your financial data. This suggests that you have a relatively healthy financial situation, with no s...
[9] USER

In [9]:
# ----------------------------------------------------------
# Cell 7 – Fresh session for a different user
# ----------------------------------------------------------
OTHER_USER = 'user_2'
session2 = create_session(OTHER_USER)
print(f'\n=== New session for {OTHER_USER} ===')
print(session2.analyze())



=== New session for user_2 ===
Based on the analysis, here are three prioritized recommendations for user 'user_2':

1. **Cut monthly expenses by 15%**: By reducing monthly expenses from $4,158.23 to $3,534.50, user 'user_2' can save an extra $623.73 per month, resulting in a total savings of $7,484.82 over 12 months. This will also increase their savings rate from 57% to 63%.
2. **Invest annual net savings at 7%**: By investing their annual net savings of $65,502.56 at a 7% interest rate, user 'user_2' can grow their portfolio to $905,012.71 over 10 years, earning a total interest of $249,987.11.
3. **Save 10% more of monthly income**: By increasing their monthly savings by 10%, user 'user_2' can save an extra $11,540.13 over 12 months, bringing their total savings to $77,940.95. This will also improve their savings rate and provide a cushion for future expenses.


In [10]:
# ----------------------------------------------------------
# Cell 8 – Clear history and re-ask in the same session
# ----------------------------------------------------------
session.clear_history()
print(f'History cleared. Turn count reset to: {session.turn_count}')

answer = session.chat('Give me a quick summary of my top 3 financial risks.')
print(answer)


History cleared. Turn count reset to: 0
Based on the provided financial data, the top 3 financial risks for user_1 are:

1. Overspending: With a total income of $165,306.82 and total expenses of $58,181.77, user_1 has a savings rate of 0.65, which is relatively high. However, the expenses are still significant, and there may be opportunities to optimize spending.
2. Low savings rate in certain categories: While the overall savings rate is 0.65, some categories like "Dining" ($5,192.37) and "Entertainment" ($5,041.22) have relatively high expenses. Reducing spending in these areas could lead to increased savings.
3. Potential for idle savings: The "Savings Transfer" category has an amount of $6,068.74, which might indicate that user_1 has some idle savings that could be invested or used more efficiently.

To mitigate these risks, user_1 could consider the following strategies:

* Review and optimize expenses in categories like "Dining" and "Entertainment" to allocate more funds towards 